# 01 -- Data Extraction

Deep Feature Synthesis over Neo4j -> weekly per-user feature matrix (`df_master`).
See the accompanying manuscript for dataset and feature background.

In [1]:
import os
import sys
sys.path.insert(0, '../src')

from awik.config import ANSWER_SET, MALICIOUS_USERS, BATCH_SIZE
from awik.neo4j_client import Neo4jConnection
from awik.features import build_master_features, ALL_FEATURES
from awik import io

db_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", os.environ.get("NEO4J_PASSWORD", "changeme"))

print(f"Answer Set: {len(MALICIOUS_USERS)} accounts (evaluation only -- see notebook 05, "
      f"never fed into Layer 1/2 before that).")
for uid, info in ANSWER_SET.items():
    print(f"  [{info['scenario']}] {uid:10s} | {info['role']:20s} | {info['description'][:45]}")

=> Neo4j: connected.
Answer Set: 6 accounts (evaluation only -- see notebook 05, never fed into Layer 1/2 before that).
  [1] ACM2278    | ATTACKER             | After-hours logon + USB + upload to wikileaks
  [2] CMP2946    | ATTACKER             | Job search + USB exfiltration to competitor
  [3] PLJ1771    | ATTACKER             | Lateral movement via keylogger -- primary att
  [3] HIS1706    | HIJACKED_ACCOUNT     | Supervisor account hijacked by PLJ1771 for ma
  [4] CDE1846    | ATTACKER             | Low-and-slow -- logs into others' machines, e
  [5] MBG3183    | ATTACKER             | Uploads documents to Dropbox after terminatio


In [2]:
df_master, weeks_df = build_master_features(db_conn, BATCH_SIZE)
print(f"Features: {len(ALL_FEATURES)} -> {ALL_FEATURES}")
print(f"Windows: {len(weeks_df)} weeks")
df_master.head(3)

  Total users: 4,000 | batch=50


  DFS batch 10/80 | 33s


  DFS batch 20/80 | 305s


  DFS batch 30/80 | 450s


  DFS batch 40/80 | 500s


...

  DFS batch 70/80 | 905s


  DFS batch 80/80 | 943s


  Checkpoint saved -> ../data/interim/dfs_checkpoint.csv


=> DFS done: 283,519 rows | 4,000 users | 951s
Features: 15 -> ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism', 'n_afterhourlogon', 'delta_afterhour', 'email_mean_size', 'delta_email_size', 'file_to_usb_bytes', 'delta_file_usb', 'file_n_delete', 'delta_file_delete', 'n_usb_connect', 'delta_usb']
Windows: 75 weeks


,User_ID,Role,Department,Openness,Conscientiousness,Extraversion,Agreeableness,Neuroticism,Year,Week,n_afterhourlogon,email_mean_size,file_to_usb_bytes,file_n_delete,n_usb_connect,delta_afterhour,delta_usb,delta_email_size,delta_file_usb,delta_file_delete
0,AAB0162,HardwareEngineer,3 - SoftwareManagement,45,36,33,15,33,2010,1,5,242448.955556,0,0,0,0.0,0.0,0.000000,0.0,0.0
1,AAB0162,HardwareEngineer,3 - SoftwareManagement,45,36,33,15,33,2010,2,5,379515.311111,0,0,0,0.0,0.0,137066.355556,0.0,0.0
2,AAB0162,HardwareEngineer,3 - SoftwareManagement,45,36,33,15,33,2010,3,5,445996.666667,0,0,0,0.0,0.0,135014.533333,0.0,0.0


In [3]:
io.save_df(df_master, '01_df_master')
io.save_df(weeks_df, '01_weeks_df')
db_conn.close()
print("Checkpoint saved -> data/interim/01_df_master.csv, 01_weeks_df.csv")

Checkpoint saved -> data/interim/01_df_master.csv, 01_weeks_df.csv
